In [ ]:
# ==============================================================================
# Setup & Dependencies
# ==============================================================================
%load_ext autoreload
%autoreload 2

import gc
import os
import sys
import gymnasium as gym
import torch

torch.set_num_threads(1)
gc.collect()

sys.path.insert(0, os.path.abspath(".."))

from config.match_config import MatchConfig, PlayerSlot, PlayerStats
from src.bots.heuristic_bot import TeamHeuristicCoordinator
from src.engine.controllers import HeuristicBotController
from src.engine.modes.classic_mode import ClassicMatchMode
from src.rl.env_wrapper import MatchEnv, PoolController, RandomController
from src.rl.ppo_core import ActorCritic
from src.rl.reset_strategies import RandomReset
from src.rl.reward_shapers import DenseReward_4
from src.rl.trainer_team import train_team_ppo

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚡ Device: {device}")

In [ ]:
# ==============================================================================
# Configuration & Directory Setup
# ==============================================================================
STAGE = 4
TEAM_SIZE = 2      # 2v2 format
NUM_ENVS = 16
MAX_STEPS = 1800   # 30.0s at 60 Hz
TIME_LIMIT = 30.0

OBS_DIM = 80       # Local actor observation
STATE_DIM = 32     # Centralized critic global state

SAVE_DIR = f"models/stage{STAGE}_phaseA"
POOL_DIR = f"models/stage{STAGE}_phaseA/pool"
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(POOL_DIR, exist_ok=True)

In [ ]:
# ==============================================================================
# Environment Factory
# ==============================================================================
def make_env(env_idx: int):
    def _init():
        import torch
        torch.set_num_threads(1)

        # 50/50 balance between Red and Blue teams across environments
        is_red = env_idx % 2 == 0
        learner_team = "red" if is_red else "blue"
        opp_team = "blue" if is_red else "red"

        # Anchored Baseline: Train strictly against 100% Heuristic bots
        opp_ctrl = HeuristicBotController(TeamHeuristicCoordinator(team=opp_team))
        #opp_ctrl = RandomController()

        """ 
        opp_ctrl = PoolController(
             pool_dir=POOL_DIR,
             team=opp_team,
             #device="cpu",
             heuristic_pct=0.4,
         )
         """

        roster = []
        for i in range(TEAM_SIZE):
            roster.append(
                PlayerSlot(
                    learner_team,
                    PlayerStats(name=f"Learner_{i}", accel=3200.0),
                    controller="RL",
                )
            )
        for i in range(TEAM_SIZE):
            roster.append(
                PlayerSlot(
                    opp_team,
                    PlayerStats(name=f"Opponent_{i}", accel=3200.0),
                    controller=opp_ctrl,
                )
            )

        cfg = MatchConfig(
            mode=ClassicMatchMode(time_limit=TIME_LIMIT, score_limit=99),
            roster=roster,
        )

        return MatchEnv(
            match_config=cfg,
            reward_shaper=DenseReward_4(team=learner_team),
            reset_strategy=RandomReset(),
            learner_team=learner_team,
            max_steps=MAX_STEPS,
        )

    return _init

# Parallelize 16 environments across CPU cores
train_envs = gym.vector.AsyncVectorEnv(
    [make_env(i) for i in range(NUM_ENVS)], 
    context="spawn"
)

In [ ]:
# ==============================================================================
# Model Initialization & Selective Weight Loading
# ==============================================================================
# Instantiate dual-tower MAPPO model: 80d Actor + 32d Critic
model = ActorCritic(obs_dim=OBS_DIM, state_dim=STATE_DIM).to(device)


stage3_best = "models/stage3/best_model.pt"
if os.path.exists(stage3_best):
    # Bootstrap Actor weights from Stage 3 while keeping the 32d Critic fresh
    model.load_actor_weights(stage3_best, device=device)
else:
    print(f"⚠️ Warning: '{stage3_best}' not found. Training from scratch.")


In [ ]:
# ==============================================================================
# MAPPO Training Loop
# ==============================================================================
train_team_ppo(
    envs=train_envs,
    model=model,
    device=device,
    team_size=TEAM_SIZE,
    baseline_type="heuristic",    
    pretrained_model_path="models/stage3/best_model.pt",
    warmup_steps=500_000,       # Freezes Actor for first 1500k steps
    lr_actor_initial=3e-5,    # Gentle fine-tuning for Stage 3 reflexes
    lr_actor_final=5e-6,
    lr_critic_initial=3e-4,     # Fast learning rate for the fresh 32d Critic
    lr_critic_final=1e-5,
    ppo_epochs_post=4,
    total_timesteps=4_500_000,
    kl_coef=0.005,
    eval_freq=100_000,
    save_dir=SAVE_DIR,
    pool_dir=POOL_DIR,
)

train_envs.close()